# 🔍 Face Recognition CNN — Google Colab Training

**Runtime → Change runtime type → GPU (T4 or A100)**

This notebook walks through:
1. Install dependencies
2. Clone / upload the project
3. Download the LFW dataset
4. Run offline face alignment
5. Train (Phase 1 warm-up + Phase 2 fine-tuning)
6. Evaluate (ROC / AUC / EER)
7. Run inference (compare two faces or webcam)

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────────────
import tensorflow as tf
print('TF version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
if not gpus:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → GPU')

In [ ]:
# ── Cell 2: Install dependencies ────────────────────────────────────────
!pip install -q ultralytics huggingface_hub scikit-learn opencv-python-headless

In [ ]:
# ── Cell 3: Upload project files ─────────────────────────────────────────
# Option A: If you have a zip of the project:
# from google.colab import files
# uploaded = files.upload()  # upload face_recognition_project.zip
# !unzip -q face_recognition_project.zip -d /content/

# Option B: Clone from your GitHub repo:
# !git clone https://github.com/YOUR_USERNAME/face_recognition_project /content/face_recognition_project

# Option C: The files were already placed by the previous cell
import os
os.chdir('/content/face_recognition_project')
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# ── Cell 4: Download LFW dataset ─────────────────────────────────────────
# Approx 180 MB, ~1680 identities with ≥5 images each
!python download_lfw.py --min-images 5

# For CASIA-WebFace (better accuracy, ~4 GB, requires manual download):
# 1. Download from https://github.com/JDAI-CV/FaceX-Zoo or academic mirrors
# 2. Extract to /content/face_recognition_project/data/faces/

In [ ]:
# ── Cell 5: Run offline face alignment ───────────────────────────────────
import sys
sys.path.insert(0, '/content/face_recognition_project')

from dataset import prepare_aligned_dataset
aligned_dir = prepare_aligned_dataset()
print('Aligned dataset:', aligned_dir)

In [ ]:
# ── Cell 6: Inspect dataset ───────────────────────────────────────────────
import os
from dataset import build_class_catalogue

import config
aligned = config.DATA_DIR.rstrip('/') + '_aligned'
data_dir = aligned if os.path.exists(aligned) else config.DATA_DIR

class_names, class_to_idx = build_class_catalogue(data_dir)
print(f'Identities: {len(class_names)}')
print('First 10:', class_names[:10])

In [ ]:
# ── Cell 7: Launch TensorBoard ────────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir /content/face_recognition_project/logs/

In [ ]:
# ── Cell 8: Train (both phases) ───────────────────────────────────────────
# This will run Phase 1 (warm-up) then Phase 2 (fine-tuning) automatically.
# Estimated time on T4: ~2-3 hours for LFW
!python train.py

In [ ]:
# ── Cell 9: View training curves ─────────────────────────────────────────
from IPython.display import Image
Image('/content/face_recognition_project/outputs/training_history.png')

In [ ]:
# ── Cell 10: Evaluate (ROC / AUC / EER) ──────────────────────────────────
!python evaluate.py

In [ ]:
# ── Cell 11: View ROC curve ───────────────────────────────────────────────
from IPython.display import Image
Image('/content/face_recognition_project/outputs/roc_curve.png')

In [ ]:
# ── Cell 12: Compare two face images ─────────────────────────────────────
# Upload your own images first:
# from google.colab import files
# uploaded = files.upload()  # upload img1.jpg and img2.jpg

from inference import compare_two_images
result = compare_two_images('img1.jpg', 'img2.jpg')

In [ ]:
# ── Cell 13: Identify faces in a photo ───────────────────────────────────
# First populate data/gallery/ with identity sub-folders.
# Then:
import cv2
from inference import FaceIdentifier
from IPython.display import Image as IpyImage

identifier = FaceIdentifier(gallery_dir='/content/face_recognition_project/data/gallery')
img_bgr = cv2.imread('group_photo.jpg')   # replace with your photo
vis = identifier.identify_and_draw(
    img_bgr,
    output_path='/content/face_recognition_project/outputs/identified.jpg'
)
IpyImage('/content/face_recognition_project/outputs/identified.jpg')

In [ ]:
# ── Cell 14: Webcam face recognition (Colab) ─────────────────────────────
from inference import run_webcam_colab
run_webcam_colab(gallery_dir='/content/face_recognition_project/data/gallery')

In [ ]:
# ── Cell 15: Download trained model ──────────────────────────────────────
from google.colab import files
files.download('/content/face_recognition_project/checkpoints/best_embedding_model.keras')